In [47]:
import os
import sys

import json
import random
import time
import pandas as pd
import seaborn as sns

from datetime import datetime
from dotenv import load_dotenv
from pathlib import Path

In [2]:
MODULES_PATH = Path("../modules").resolve()

if str(MODULES_PATH) not in sys.path:
    sys.path.insert(0, str(MODULES_PATH))

import corpus
import api

from dev import rel, print_epi_summary
from data_config import DATA_CONFIG, FEW_SHOT_PATH
from few_shot import get_few_shot_examples
from error_sampling import error_summary, sample_errors

import pipeline

# Config.

### Development Config.

In [3]:
load_dotenv()

# ---------- Set seed to 42 ----------
SEED = int(os.getenv("SEED", 42))
random.seed(SEED)

# ---------- Print confirmation ----------
print(
    f"Set random seed to {SEED} at "
    f"{datetime.now().strftime('%Y-%m-%d %H:%M')}"
)

Set random seed to 42 at 2026-08-09 20:15


### EPI Config.

In [4]:
# ---------- Manual EPI config. entry ----------
EPI_NUM = "007"
DATASET_SPLIT = "train"

# ---------- Config whether EPI config's API should be called again ----------
CALL_API = True

# Corpus Loading

### Load Corpus

In [5]:
# ---------- Get abstracts path ----------
ABSTRACTS_PATH = DATA_CONFIG[DATASET_SPLIT]["paths"]["abstracts"]
SAMPLE_SIZE = DATA_CONFIG[DATASET_SPLIT]["sample_size"]

# ---------- Load corpus from path ----------
abstracts_corpus = corpus.load_corpus(ABSTRACTS_PATH, sample_size=SAMPLE_SIZE)
print(f"Abstracts dataset length: {len(abstracts_corpus)}")

Abstracts dataset length: 35


### Get Ground Truths

In [6]:
# ---------- Manually Set Ground Truth Path ----------
GT_PATH = DATA_CONFIG[DATASET_SPLIT]["paths"]["ground_truths"]

# ---------- Fetch BioRED Ground Truths ----------
abstracts_ground_truths = corpus.get_corpus_gt_csv(abstracts_corpus, GT_PATH)

In [7]:
# ---------- Turn ground truth DataFrame into structured dict ----------
ground_truths = corpus.get_gt_dict(abstracts_ground_truths)

In [8]:
import pandas as pd
gt_df = pd.DataFrame(list(ground_truths["relations"]))
print(list(gt_df[2].unique()))

['Negative_Correlation', 'Association', 'Positive_Correlation', 'Bind', 'Comparison']


### Few-Shot Construction

In [9]:
# ---------- If FS block does not exist, create FS block, else pass ----------
if not os.path.exists(FEW_SHOT_PATH):

    few_shot_block = get_few_shot_examples(
        path_to_train_set=DATA_CONFIG["train"]["paths"]["abstracts"],
        path_to_train_gts=DATA_CONFIG["train"]["paths"]["ground_truths"],
        biored_train_samples=abstracts_corpus,
        few_shot_export_path=FEW_SHOT_PATH
    )
    print(f"Generated and exported new few-shot block to '{FEW_SHOT_PATH}'")
else:
    with open(FEW_SHOT_PATH) as f:
        few_shot_block = f.read()
    print(f"Imported existing few-shot block from '{FEW_SHOT_PATH}' at {datetime.now().strftime('%Y-%m-%d %H:%M')}.")

Imported existing few-shot block from '../../data/few_shot/few_shot_block.txt' at 2026-08-09 20:15.


### Import BioRED Extraction Guidelines

In [10]:
# ---------- Import BioRED guidelines text file for prompt refinement ----------
with open("../../data/processed/biored/guidelines.txt", "r", encoding="utf-8") as f:
    biored_ext_guidelines = f.read()

# ---------- Print preview ----------
print(f"{biored_ext_guidelines[:500]}...")

## Guideline of the entities

### General rules
- Annotate all the spans of all the six concept types.
- The full text can be accessed to clarify the concept spans and identifiers.
- The abbreviation and its long form should be annotated separately if possible. prostaglandin E2 (PGE2) in the text, “prostaglandin E2” and “PGE2” should be both annotated to chemicals with the same identifier (D015232).
- Annotate both the full name and abbreviation in one entity, if the boundary of the entity cover...


# OpenAI Luna API Call

### EPI Setup

In [11]:
# ---------- Create EPI setup dictionary ----------
# - Keys: "dataset", "id", "eval_version", "notes", "reuse_api_call", "prompt"
epi_setup = pipeline.setup_epi(EPI_NUM, DATASET_SPLIT)

# ---------- Print summary ----------
print_epi_summary(
    epi_num=EPI_NUM,
    prompt_version=epi_setup["prompt"]["version"],
    eval_version=epi_setup["eval_version"],
    notes=epi_setup["notes"],
    reuse_api_call=epi_setup["reuse_api_call"]
)

Cell ran at 2026-08-09 20:15 for epi_007
 - Prompt version: v4
 - Evaluation version: v4
 - Notes: Refined prompt to avoid use of background biomedical knowledge, co-occurence rules & included explicit formatting.
 - Reuse API Call: False


### API Call

In [12]:
if CALL_API:
    # ---------- Create client ----------
    client = api.create_client()

    # ---------- If prompt version is different from previous EPI, call API, else pass ----------
    if not epi_setup["reuse_api_call"]:

        print(f"Running API call for {epi_setup["id"]}...")

        # Fetch raw prompt template
        prompt_template = epi_setup["prompt"]["template"]

        # Begin timer
        start_time = time.perf_counter()

        # Initiate outputs list
        outputs = []

        # Begin looping through abstract dataset rows - one call per row
        for index, row in abstracts_corpus.iterrows():
            abstract = row["abstract"]

            # Replace prompt template's placeholders with abstract, few_shot & guidelines
            prompt = (
                prompt_template
                .replace("{abstract}", abstract)
                .replace("{few_shot_block}", few_shot_block)
                .replace("{biored_ext_guidelines}", biored_ext_guidelines)
            )

            # Store row's response
            response = client.responses.create(
                model="gpt-5.6-luna",
                input=prompt
            )

            # Append response to outputs list
            outputs.append({
                "pmid": row["pmid"],
                "output": response.output_text
            })

        # End time, store elapsed time & print result
        elapsed_seconds = time.perf_counter() - start_time
        print(f"API calls took {elapsed_seconds:.1f}s ({elapsed_seconds/60:.1f} min) for {len(outputs)} abstracts\n")

        print(f"Successfully ran API call at {datetime.now().strftime('%Y-%m-%d %H:%M')}")
        print(f" - Prompt verion: {epi_setup["prompt"]["version"]}")
        print(f" - Evalaution verion: {epi_setup["eval_version"]}")
        print(f" - Run notes: {epi_setup["notes"]}")
        print(f" - Elapsed time: {elapsed_seconds:.1f}s ({elapsed_seconds/60:.1f}")
    else:
        prev_epi_id = f"epi_{int(EPI_NUM) - 1:03d}"

        with open(f"{DATA_CONFIG[DATASET_SPLIT]["paths"]["epis"]}/{prev_epi_id}.json") as f:
            prev_epi_log = json.load(f)

        outputs = prev_epi_log["outputs"]
        prompt_template = prev_epi_log["prompt"]
        elapsed_seconds = prev_epi_log["time_taken"]

        print(f"Reused API call from {prev_epi_id}.")

    output_info = {
        "epi_id": epi_setup["id"],
        "outputs": outputs,
        "time_taken": elapsed_seconds,
        "raw_prompt": prompt_template,
        "epi_notes": epi_setup["notes"],
        "prompt_version": epi_setup["prompt"]["version"],
        "eval_version": epi_setup["eval_version"],
        "export_path": DATA_CONFIG[DATASET_SPLIT]["paths"]["epis"]
    }
else:
    print(f"Pipeline run not executed at {datetime.now().strftime('%Y-%m-%d %H:%M')} (PIPELINE_RUN = False).")

Created client:
 - Timeout: 60
 - Max retries: 0

Running API call for epi_007...
API calls took 563.5s (9.4 min) for 35 abstracts

Successfully ran API call at 2026-08-09 20:24
 - Prompt verion: v4
 - Evalaution verion: v4
 - Run notes: Refined prompt to avoid use of background biomedical knowledge, co-occurence rules & included explicit formatting.
 - Elapsed time: 563.5s (9.4


In [13]:
# ---------- Parse outputs, evaluate extractions, export EPI results ----------
if CALL_API:
    epi_log = pipeline.process_epi(output_info, ground_truths, dataset=epi_setup["dataset"])
else:
    print(f"Pipeline run not executed at {datetime.now().strftime('%Y-%m-%d %H:%M')} (CALL_API = False).")

Parsed 35 extractions, 0 failed to parse as JSON


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Saved epi_007 to ../../data/results/epis/biored_train


# Error Analysis

### Summary Table

In [14]:
ERROR_SAMPLE_SIZE = 20

In [37]:
# ---------- If EPI API call does not exist in kernel state, import existing EPI log ----------
if not CALL_API:
    try:
        with open(f"../../data/results/epis/{epi_setup["dataset"]}/{epi_setup["id"]}.json") as f:
            epi_log = json.load(f)
    except:
        raise ValueError(
            f"EPI log cannot be imported: "
            f"'../../data/results/epis/{epi_setup["dataset"]}/{epi_setup["id"]}.json' does not exist."
        )

error_summary(epi_log)

`epi_007` Error Summary:


Metric,Precision,Recall,F1
Entity,0.582,0.949,0.721
Relation,0.473,0.564,0.515


### Distribution Comparison

In [38]:
# Access ground truth relation distributions
# Access epi_log ground truth relation distributions
# Plot

In [53]:
gt_relations = ground_truths["relations"]

gt_relation_counts = pd.Series([r[2] for r in gt_relations]).value_counts()

In [ ]:
from parsing import outputs_to_extractions, extractions_to_tuples

# ---------- Get EPI's outputs and parse to sets of tuples ----------
# Extractions: str -> list[dicts]
extractions, _ = outputs_to_extractions(epi_log["outputs"])

# Extractions: dict -> set{tuples}
_, predictions_relations = extractions_to_tuples(extractions)

relation_counts = pd.Series([r[2] for r in predictions_relations]).value_counts()

Parsed 35 extractions, 0 failed to parse as JSON


## Relations

### False Positives

In [26]:
import error_sampling
rel(error_sampling)
from error_sampling import sample_errors

Reloaded 'error_sampling' module at 2026-08-09 20:45.


In [36]:
# ---------- False positives ----------
relations_fp = epi_log["errors"]["errors"]["relations"]["false_positives"]

print(f"Count: {len(relations_fp)}")

relations_fp_errors = sample_errors(relations_fp, sample_size=200)
pd.DataFrame(relations_fp_errors)[2].value_counts()

Count: 186


2
Positive_Correlation    76
Association             46
Negative_Correlation    45
Bind                    11
Comparison               6
Cotreatment              2
Name: count, dtype: int64

### False Negatives

In [18]:
# ---------- False negatives ----------
relations_fn = epi_log["errors"]["errors"]["relations"]["false_negatives"]

print(f"Count: {len(relations_fn)}")

sample_errors(relations_fn)

Count: 129
Sample size: 20


[[28512644, 'ccl11', 'Association', 'erysipelas'],
 [24743235, 'csf-1', 'Association', 'il-3'],
 [18827003,
  'aspartic acid to histidine substitution at amino acid position 401',
  'Association',
  'dexamethasone'],
 [18827003, 'glucocorticoid', 'Association', 'cortisol'],
 [28512644, 'cxcl9', 'Association', 'erysipelas'],
 [29222418, 'ifng', 'Association', 'heb'],
 [28512644, 'il-10', 'Association', 'erysipelas'],
 [18768591,
  'urea',
  'Association',
  'serum- and glucocorticoid-inducible kinase 1'],
 [16288197, 'dct', 'Association', 'pigmentation'],
 [29222418, 'vg4', 'Association', 'heb'],
 [19108278, 'isoprenaline', 'Positive_Correlation', 'glucose'],
 [28428256, 'ep4', 'Positive_Correlation', 'rap1/rac1 gtpase'],
 [28512644, 'il-2ralpha', 'Association', 'erysipelas'],
 [16288197, 'cyp1b1', 'Association', 'glaucoma'],
 [24341598, 'sodium bicarbonate', 'Comparison', 'diltiazem'],
 [18827003,
  '(g --> c) substitution at position 1201',
  'Positive_Correlation',
  'glucocorticoid'

## Entities

### False Positives

In [19]:
# ---------- False positives ----------
entities_fp = epi_log["errors"]["errors"]["entities"]["false_positives"]

print(f"Count: {len(entities_fp)}")

sample_errors(entities_fp)

Count: 174
Sample size: 20


[[21163864, 'left ventricular hypertrophy', 'DiseaseOrPhenotypicFeature'],
 [19918264, 'prostate cancer', 'DiseaseOrPhenotypicFeature'],
 [21163864,
  'sporadic hypertrophic cardiomyopathy',
  'DiseaseOrPhenotypicFeature'],
 [24914936, 'tsh', 'ChemicalEntity'],
 [29222418, 'cd24', 'GeneOrGeneProduct'],
 [19108278, 'isoprenaline', 'ChemicalEntity'],
 [15099351,
  'proprotein convertase subtilisin/kexin type 9',
  'GeneOrGeneProduct'],
 [19521089, 'slc6a4', 'GeneOrGeneProduct'],
 [21163864,
  'methionine to threonine substitution at codon 235',
  'SequenceVariant'],
 [21771880, 'fat mass', 'DiseaseOrPhenotypicFeature'],
 [16200390, 'tryptophan hydroxylase', 'GeneOrGeneProduct'],
 [25305591, 'vpac1', 'GeneOrGeneProduct'],
 [24743235, 'il-3', 'GeneOrGeneProduct'],
 [19108278, 'lactate', 'ChemicalEntity'],
 [20683499, 'crocin', 'ChemicalEntity'],
 [16200390, '5-ht', 'ChemicalEntity'],
 [21163864,
  'familial hypertrophic cardiomyopathy',
  'DiseaseOrPhenotypicFeature'],
 [28411266, 'abcc8',

### False Negatives

In [20]:
# ---------- False negatives ----------
entities_fn = epi_log["errors"]["errors"]["entities"]["false_negatives"]

print(f"Count: {len(entities_fn)}")

sample_errors(entities_fn)

Count: 13


[[28348168, 'cancer', 'DiseaseOrPhenotypicFeature'],
 [18768591, 'salt', 'ChemicalEntity'],
 [28411266, 'sulfonylurea receptor', 'GeneOrGeneProduct'],
 [16288197, 'pigmentation', 'DiseaseOrPhenotypicFeature'],
 [28411266, 'insulin', 'GeneOrGeneProduct'],
 [20510337, 'lipid', 'ChemicalEntity'],
 [18768591, 'weight gain', 'DiseaseOrPhenotypicFeature'],
 [28348168, 'degenerative disease', 'DiseaseOrPhenotypicFeature'],
 [15686794, 'antiarrhythmic drug', 'ChemicalEntity'],
 [28428256, 'thrombin receptor', 'GeneOrGeneProduct'],
 [20683499, 'neurodegenerative diseases', 'DiseaseOrPhenotypicFeature'],
 [15970799, 'tritium', 'ChemicalEntity'],
 [15099351, 'cholesterol', 'ChemicalEntity']]